# Semana 12: RAG (Retrieval-Augmented Generation) y Bases de Datos Vectoriales
## Notebook Conceptual (NB1) – Búsqueda Semántica con FAISS

**Propósito:** Diseñar sistemas que combinan recuperación de información con generación para responder preguntas sobre documentos propios.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Generar embeddings de documentos usando Sentence Transformers.
- Construir un índice FAISS y almacenar los embeddings.
- Buscar documentos relevantes dada una pregunta.
- Usar un modelo generativo (GPT-2) para responder basado en los documentos recuperados.
- Evaluar la relevancia de los documentos recuperados.

---

## 0. Configuración Inicial

Importamos las librerías necesarias e instalamos dependencias.

In [ ]:
# Instalamos dependencias
!pip install sentence-transformers faiss-cpu transformers

# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import warnings
warnings.filterwarnings('ignore')

# Sentence Transformers para embeddings
from sentence_transformers import SentenceTransformer

# FAISS para búsqueda vectorial
import faiss

# Transformers para generación
from transformers import pipeline, GPT2Tokenizer, GPT2LMHeadModel

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Verificar dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

print("\nLibrerías importadas correctamente.")

---
## 1. Conjunto de Documentos (Fuente de Conocimiento)

Creamos un conjunto pequeño de documentos sobre diversos temas.

In [ ]:
# Lista de documentos (párrafos cortos)
documents = [
    "El cambio climático es causado por el aumento de gases de efecto invernadero como el CO2.",
    "Los transformers son una arquitectura de deep learning basada en mecanismos de atención.",
    "El machine learning permite a las computadoras aprender sin ser programadas explícitamente.",
    "Las redes neuronales convolucionales (CNN) son muy efectivas para el procesamiento de imágenes.",
    "El procesamiento del lenguaje natural (NLP) combina lingüística e inteligencia artificial.",
    "Python es un lenguaje de programación muy popular en ciencia de datos.",
    "La inteligencia artificial general (AGI) se refiere a máquinas con capacidades cognitivas humanas.",
    "El aprendizaje supervisado utiliza datos etiquetados para entrenar modelos.",
    "Los embeddings convierten palabras en vectores numéricos capturando significado semántico.",
    "La atención multi-head es un componente clave de los transformers."
]

# Crear DataFrame para mejor visualización
df_docs = pd.DataFrame({'id': range(len(documents)), 'documento': documents})
print("=== DOCUMENTOS ===")
df_docs

---
## 2. Generación de Embeddings con Sentence Transformers

Usamos `all-MiniLM-L6-v2`, un modelo ligero y efectivo para embeddings de oraciones.

In [ ]:
# Cargar modelo de embeddings
print("Cargando modelo de Sentence Transformers...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generar embeddings para todos los documentos
document_embeddings = embedding_model.encode(documents)

print(f"Forma de los embeddings: {document_embeddings.shape}")
print(f"(num_documentos, dimension_embedding)")

# Mostrar ejemplo
print(f"\nEmbedding del primer documento (primeras 10 dimensiones):")
print(document_embeddings[0][:10])

---
## 3. Construcción del Índice FAISS

FAISS (Facebook AI Similarity Search) es una librería para búsqueda eficiente de similitud en espacios vectoriales.

In [ ]:
# Obtener dimensión de los embeddings
dimension = document_embeddings.shape[1]

# Crear índice FAISS (usamos IndexFlatL2 para búsqueda exacta por distancia L2)
index = faiss.IndexFlatL2(dimension)

# Verificar que el índice esté vacío
print(f"Número de vectores en el índice: {index.ntotal}")

# Añadir embeddings al índice
index.add(document_embeddings.astype('float32'))

print(f"Número de vectores después de añadir: {index.ntotal}")

# Opcional: Guardar índice en disco
faiss.write_index(index, "document_index.faiss")
print("Índice guardado como 'document_index.faiss'")

---
## 4. Búsqueda de Documentos Relevantes

Dada una pregunta, generamos su embedding y buscamos los documentos más cercanos.

In [ ]:
def search_documents(query, embedding_model, index, documents, k=3):
    """
    Busca los k documentos más similares a la consulta.
    """
    # Generar embedding de la consulta
    query_embedding = embedding_model.encode([query]).astype('float32')
    
    # Buscar en el índice FAISS
    distances, indices = index.search(query_embedding, k)
    
    # Mostrar resultados
    print(f"\nConsulta: {query}")
    print(f"Top {k} documentos relevantes:")
    for i, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        print(f"\n{i+1}. [Distancia: {dist:.4f}]")
        print(f"   {documents[idx]}")
    
    return indices[0], distances[0]

# Probar con algunas preguntas
queries = [
    "¿Qué causa el cambio climático?",
    "¿Qué son los transformers en deep learning?",
    "¿Qué es el aprendizaje supervisado?",
    "¿Qué lenguaje se usa en ciencia de datos?"
]

for query in queries:
    search_documents(query, embedding_model, index, documents, k=2)
    print("-"*80)

---
## 5. Generación de Respuestas con Modelo Generativo

Usamos GPT-2 para generar respuestas basadas en los documentos recuperados.

In [ ]:
# Cargar modelo generativo (GPT-2)
print("Cargando GPT-2 para generación...")
generator = pipeline('text-generation', model='gpt2', device=0 if torch.cuda.is_available() else -1)

def generate_answer(query, documents, indices, generator, max_length=100):
    """
    Genera una respuesta usando los documentos recuperados como contexto.
    """
    # Construir contexto con los documentos relevantes
    context = " ".join([documents[idx] for idx in indices])
    
    # Construir prompt
    prompt = f"""Contexto: {context}

Pregunta: {query}

Respuesta basada en el contexto:"""
    
    # Generar respuesta
    response = generator(prompt, max_length=max_length, num_return_sequences=1, temperature=0.7)[0]['generated_text']
    
    # Extraer solo la respuesta (después del prompt)
    answer = response[len(prompt):].strip()
    
    return answer

# Probar generación para una consulta
query = "¿Qué causa el cambio climático?"
indices, distances = search_documents(query, embedding_model, index, documents, k=2)
answer = generate_answer(query, documents, indices, generator)

print(f"\n=== RESPUESTA GENERADA ===")
print(answer)

In [ ]:
# Probar con más preguntas
test_queries = [
    "¿Qué son los transformers?",
    "¿Qué es machine learning?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    indices, distances = search_documents(query, embedding_model, index, documents, k=2)
    answer = generate_answer(query, documents, indices, generator, max_length=80)
    print(f"\nRESPUESTA: {answer}")

---
## 6. Evaluación de la Relevancia de Documentos Recuperados

Evaluamos manualmente si los documentos recuperados son relevantes para la pregunta.

In [ ]:
def evaluate_relevance(query, documents, indices):
    """
    Evaluación manual de relevancia (simulada).
    En un sistema real, se usarían métricas como Precision@k o NDCG.
    """
    print(f"Pregunta: {query}")
    print("Documentos recuperados:")
    for i, idx in enumerate(indices):
        print(f"  {i+1}. {documents[idx]}")
    
    # Aquí el usuario evaluaría manualmente (simulamos con una nota)
    # En un sistema real, se compararía con un conjunto de relevancia etiquetado
    relevance_score = input("\nPuntaje de relevancia (1-5) para el conjunto: ")
    return int(relevance_score)

# Probar evaluación
query_eval = "¿Qué son los transformers?"
indices_eval, _ = search_documents(query_eval, embedding_model, index, documents, k=3)
# Descomentar para evaluar interactivamente
# score = evaluate_relevance(query_eval, documents, indices_eval)
# print(f"Puntaje de relevancia: {score}/5")

### 6.1. Métricas Automáticas de Relevancia

Podemos usar las distancias FAISS como proxy de relevancia.

In [ ]:
def analyze_distances(query, embedding_model, index, documents, k=5):
    """
    Analiza las distancias de los documentos recuperados.
    """
    query_embedding = embedding_model.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, k)
    
    print(f"Pregunta: {query}")
    print("Distancias de los documentos recuperados:")
    for i, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        print(f"  {i+1}. Distancia: {dist:.4f} - {documents[idx][:50]}...")
    
    print(f"\nDistancia media: {np.mean(distances[0]):.4f}")
    print(f"Desviación estándar: {np.std(distances[0]):.4f}")
    
    # Visualizar distribución de distancias
    plt.figure(figsize=(8, 4))
    plt.bar(range(1, k+1), distances[0])
    plt.xlabel('Documento (ranking)')
    plt.ylabel('Distancia')
    plt.title('Distancias de documentos recuperados')
    plt.show()
    
    return distances, indices

analyze_distances("¿Qué es el aprendizaje supervisado?", embedding_model, index, documents)

---
## 7. Sistema RAG Completo

Integramos todos los pasos en una función única.

In [ ]:
def rag_system(query, embedding_model, index, documents, generator, k=3, max_length=100):
    """
    Sistema RAG completo.
    """
    # 1. Recuperar documentos relevantes
    query_embedding = embedding_model.encode([query]).astype('float32')
    distances, indices = index.search(query_embedding, k)
    
    # 2. Construir contexto
    context = " ".join([documents[idx] for idx in indices[0]])
    
    # 3. Generar respuesta
    prompt = f"Contexto: {context}\n\nPregunta: {query}\n\nRespuesta basada en el contexto:"
    response = generator(prompt, max_length=max_length, num_return_sequences=1, temperature=0.7)[0]['generated_text']
    answer = response[len(prompt):].strip()
    
    # 4. Mostrar resultados
    print(f"\n{'='*60}")
    print(f"Pregunta: {query}")
    print(f"\nDocumentos recuperados:")
    for i, idx in enumerate(indices[0]):
        print(f"  {i+1}. {documents[idx]}")
    print(f"\nRespuesta generada:")
    print(answer)
    
    return answer, indices[0], distances[0]

# Probar el sistema completo
rag_system("¿Qué son los transformers?", embedding_model, index, documents, generator)

---
## 8. Ejercicio para el Estudiante

1. **Amplía la base de documentos**: Añade más documentos sobre diferentes temas y observa cómo mejora el sistema.

2. **Cambia el modelo de embeddings**: Prueba con `'all-mpnet-base-v2'` (más preciso) y compara resultados.

3. **Usa otro modelo generativo**: Prueba con `'gpt2-medium'` o `'gpt2-large'` para respuestas más detalladas.

4. **Implementa búsqueda con umbral**: Solo recupera documentos con distancia menor a un umbral.

5. **Evalúa con preguntas sin respuesta**: ¿Qué pasa si la pregunta no tiene documentos relevantes?

In [ ]:
# Espacio para el estudiante
# ...

---
## 9. Conclusiones

En este notebook hemos implementado un sistema RAG completo:

✔️ **Embeddings**: Generamos representaciones vectoriales de documentos usando Sentence Transformers.

✔️ **Índice FAISS**: Construimos un índice para búsqueda eficiente de similitud.

✔️ **Recuperación**: Dada una pregunta, encontramos los documentos más relevantes.

✔️ **Generación**: Usamos GPT-2 para generar respuestas basadas en el contexto recuperado.

✔️ **Evaluación**: Analizamos las distancias como proxy de relevancia.

**Lección clave**: RAG permite combinar el conocimiento de un LLM con información específica y actualizada, reduciendo alucinaciones y permitiendo aplicaciones sobre documentos privados.

---
**Fin del Notebook Conceptual - Semana 12 (NLP)**